# Neighbour label counts for cell typing (`future_plan.md` §16, arms A and B; pre-registered 2026-11-05)

**Base:** arm 0 of `multiscale-audit` (Table + nbr from image-derived means). Same inputs, splits and seeds, so its macro-F1 must match that run.

**Count features (DOI style):** for each class, the number of neighbours *with* the label, the number *without* it, and the number of neighbours with a known label. They are computed on the contact graph and on the 30 µm graph (contact ∪ non-contact).

**Arm A, predicted labels** (works on unseen images and cohorts):
- **A1:** + counts from out-of-fold predicted coarse types (5 folds by image on the training images; a full model elsewhere);
- **A2:** A1 + mean predicted probabilities over the same two graphs;
- **A3, round 2** (coarse only): counts and mixtures recomputed from out-of-fold A2 predictions.

**Arm B, revealed true labels** (a separate setting, reported apart):
- reveal a random fraction f of cells per image (coarse: f = 5 / 20 / 50%; fine: 20%), in training and test images alike;
- counts are taken over revealed neighbours only;
- training and scoring use hidden cells only, paired with arm 0 on the same hidden cells;
- transfer units reveal the target cohort's labels.

**Settings:** coarse J→J, D→D, J→D, D→J; fine D→D, J→J. 3 replicates; macro-F1 with a 1000× patient bootstrap.

**Claim rule:** Δ macro-F1 vs arm 0 has mean ≥ +0.01 and CI > 0 in 3/3 replicates.

In [ ]:
import os, glob, json, time, sys, subprocess, warnings, gc
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
from scipy.spatial import cKDTree
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm"], check=True)
    import lightgbm as lgb
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1"
SEED = 20261013                                   # same seed and split rules as cell-type-graph / train-test-grid
N_BOOT = 100 if SMOKE else 1000
COF = 5.0
N_FIT_IMP = 10_000 if SMOKE else 100_000          # training cells per imputation model
N_FIT_TYP = 10_000 if SMOKE else 150_000          # labelled training cells per typing model
N_FIT_S1 = 5_000 if SMOKE else 100_000            # stage-1 (predicted-type) models
LGB = dict(n_estimators=30 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=os.cpu_count(), verbose=-1)
BUDGET_MIN = float(os.environ.get("BUDGET_MIN", 660))
REPS = [0] if SMOKE else [0, 1, 2]
R_CAP = 32
NUC = ["histone_h3", "dna1", "dna2"]
MARKERS = sorted(["c_erb_b_2_her2", "cd20", "cd3", "cd45", "cd68", "cytokeratin_5", "cytokeratin_8_18",
                  "estrogen_receptor_alpha", "ki_67", "pan_cytokeratin", "sma"])
IN_SET = NUC + MARKERS                            # 14 channels; + log area = 15 base inputs
COARSE = ["epithelial", "T cell", "B cell", "myeloid", "endothelial", "stroma"]
D_COARSE = {"CD8^{+} T cells": "T cell", "CD4^{+} T cells": "T cell", "T_{Reg} & T_{Ex}": "T cell",
            "B cells": "B cell",
            "Macrophages": "myeloid", "Granulocytes": "myeloid", "Macrophages & granulocytes": "myeloid",
            "Endothelial": "endothelial",
            "Fibroblasts": "stroma", "Fibroblasts FSP1^{+}": "stroma", "Myofibroblasts": "stroma",
            "Myofibroblasts PDPN^{+}": "stroma"}
J_COARSE = {1: "T cell", 2: "B cell", 4: "myeloid", 6: "myeloid", 7: "endothelial",
            **{m: "stroma" for m in range(8, 14)}, **{m: "epithelial" for m in range(14, 28)}}
T0 = time.time()

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def near(name, anchor):
    hits = g(name)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

In [ ]:
P = {}
for p in g("cells.parquet"):
    key = "Danenberg" if "metabric_id" in pq.read_schema(p).names else "Jackson"
    P[key] = dict(cells=p, contacts=near("cell_neighbours.parquet", p),
                  pix=g(f"pixfeat_{key.lower()}.parquet")[0])
P["Danenberg"]["split"] = near("danenberg_patient_split.csv", P["Danenberg"]["cells"])
print(json.dumps(P, indent=1))

def harm(X):
    X = np.asarray(X, np.float64)
    med = np.nanmedian(X, 0)
    q05, q95 = np.nanpercentile(X, [5, 95], 0)
    sc = np.where(q95 - q05 > 1e-3, q95 - q05, np.nanstd(X, 0) + 1e-6)
    return ((X - med) / sc).astype(np.float32)

def load(key):
    pf = pd.read_parquet(P[key]["pix"])
    cells = pd.read_parquet(P[key]["cells"])
    cells = cells.set_index(["image_name", "object_number"]).loc[
        pd.MultiIndex.from_arrays([pf.image_name.values, pf.object_number.values])].reset_index()
    n = len(cells)
    img, img_names = pd.factorize(cells.image_name, sort=True)
    assert (np.diff(img) >= 0).all()
    starts = np.r_[0, np.flatnonzero(np.diff(img)) + 1, n]
    loc = pd.Series(np.arange(n), index=pd.MultiIndex.from_arrays([cells.image_number.values, cells.object_number.values]))
    con = pd.read_parquet(P[key]["contacts"])
    ca = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["from"].values])).values
    cb = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["to"].values])).values
    ok = ~(np.isnan(ca) | np.isnan(cb))
    ca, cb = ca[ok].astype(np.int64), cb[ok].astype(np.int64)
    ca, cb = ca[ca != cb], cb[ca != cb]
    CON = sp.coo_matrix((np.ones(len(ca), np.float32), (ca, cb)), shape=(n, n)).tocsr()
    CON = ((CON + CON.T) > 0).astype(np.float32).tocsr()
    phen = cells.cell_phenotype.astype(str).values
    fine = sorted(set(phen))
    y_fine = pd.Series(phen).map({p: i for i, p in enumerate(fine)}).values.astype(np.int64)
    if key == "Danenberg":
        epi = cells.groupby(cells.cell_phenotype.astype(str)).is_epithelial.mean()
        cmap = {p: ("epithelial" if epi[p] > 0.5 else D_COARSE.get(p)) for p in fine}
        pid = "metabric_id"
    else:
        mc = cells.groupby(cells.cell_phenotype.astype(str)).metacluster.first()
        cmap = {p: J_COARSE.get(int(mc[p])) for p in fine}
        pid = "patient_id"
    coarse_of = np.array([COARSE.index(cmap[p]) if cmap[p] else -1 for p in fine], np.int64)
    X15 = np.c_[np.arcsinh(pf[[f"raw__{c}" for c in IN_SET]].values.astype(np.float64) / COF),
                np.log(np.maximum(cells["area"].values, 1))]
    F = {"base": np.nan_to_num(harm(X15))}
    pix = lambda v: [f"{s}_{v}__{c}" for c in IN_SET for s in ("dio", "sd", "mi")]
    shape = ["ecc", "perim", "dmax"]
    F["pix"] = harm(pf[pix("full") + shape].values)
    F["pix_e2"] = harm(pf[pix("e2") + shape].values)
    F["band"] = harm(pf[[f"band__{c}" for c in IN_SET]].values)
    names = {"pix": pix("full") + shape, "pix_e2": pix("e2") + shape, "band": [f"band__{c}" for c in IN_SET]}
    return dict(key=key, n=n, starts=starts, img=img.astype(np.int64), n_img=len(img_names), img_names=np.asarray(img_names),
                patient=cells[pid].astype(str).values, xy=cells[["x", "y"]].values.astype(np.float64),
                CON=CON, fine=fine, y={"fine": y_fine, "coarse": coarse_of[y_fine]}, F=F, names=names)

CO = {k: load(k) for k in ("Jackson", "Danenberg")}
CJ, CD = CO["Jackson"], CO["Danenberg"]
split0 = pd.read_csv(P["Danenberg"]["split"]).set_index("metabric_id").split
CD["split0"] = pd.Series(CD["patient"]).map(split0).fillna("none").values
for k, C in CO.items():
    print(f"{k}: {C['n']:,} cells, {C['n_img']} images, {len(set(C['patient']))} patients; "
          f"pixel-feature NaN fraction {np.isnan(C['F']['pix']).mean():.3f} (full) / {np.isnan(C['F']['pix_e2']).mean():.3f} (e2)")
tick("tables and pixel features loaded")

In [ ]:
def adjacency(C, kind, par):
    rows, cols = [], []
    for a, b in zip(C["starts"][:-1], C["starts"][1:]):
        xy = C["xy"][a:b]
        k = min(par if kind == "knn" else R_CAP, b - a - 1)
        if kind == "knn":
            d, nb = cKDTree(xy).query(xy, k=k + 1)
        else:
            d, nb = cKDTree(xy).query(xy, k=k + 1, distance_upper_bound=par)
        d, nb = d[:, 1:], nb[:, 1:]
        ok = np.isfinite(d)
        rows.append(np.repeat(np.arange(a, b), k)[ok.ravel()])
        cols.append(nb[ok] + a)
    rows, cols = np.concatenate(rows), np.concatenate(cols)
    A = sp.coo_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(C["n"], C["n"])).tocsr()
    A = ((A + A.T) > 0).astype(np.float32).tocsr()
    A.sort_indices()
    return A

def row_mean(A, X):
    deg = np.asarray(A.sum(1)).ravel()
    return np.where(deg[:, None] > 0, (A @ X) / np.maximum(deg, 1)[:, None], np.nan).astype(np.float32), deg

def grid_mean(C, X, size):
    """Mean of X over the cells in the same size-µm grid square of the same image, and the square's cell count."""
    gx = np.floor(C["xy"][:, 0] / size).astype(np.int64)
    gy = np.floor(C["xy"][:, 1] / size).astype(np.int64)
    key = (C["img"] * 100_000 + gy) * 100_000 + gx
    code, inv = np.unique(key, return_inverse=True)
    cnt = np.bincount(inv).astype(np.float64)
    M = np.stack([np.bincount(inv, X[:, j], len(code)) for j in range(X.shape[1])], 1) / cnt[:, None]
    return M[inv].astype(np.float32), np.log1p(cnt[inv]).astype(np.float32)

for key, C in CO.items():
    X = C["F"]["base"]
    A10 = adjacency(C, "knn", 10)
    A30 = adjacency(C, "radius", 30.0)
    deg = np.diff(A10.indptr).astype(np.float32)[:, None]
    mean = (A10 @ X) / deg
    std = np.sqrt(np.maximum((A10 @ (X * X)) / deg - mean * mean, 0))
    mx = np.maximum.reduceat(X[A10.indices], A10.indptr[:-1], axis=0)
    hop2 = (A10 @ mean) / deg
    dst = np.repeat(np.arange(C["n"]), np.diff(A10.indptr))
    dist = np.bincount(dst, np.linalg.norm(C["xy"][A10.indices] - C["xy"][dst], axis=1), C["n"]) / deg.ravel()
    C["F"]["nbr_blocks"] = np.stack([mean, mx, std, hop2], 0).astype(np.float32)          # 4 x n x 15
    C["F"]["nbr_extra"] = np.c_[np.log1p(np.diff(A30.indptr)), np.log1p(np.diff(C["CON"].indptr)), dist / 30.0].astype(np.float32)
    CON30 = A30.multiply(C["CON"]).tocsr()
    NON30 = (A30 - CON30).tocsr()
    NON30.eliminate_zeros()
    C["A_con"], C["A_non"] = C["CON"], NON30
    m_con, _ = row_mean(C["CON"], X)
    m_non, _ = row_mean(NON30, X)
    g64, n64 = grid_mean(C, X, 64.0)
    g128, n128 = grid_mean(C, X, 128.0)
    C["F"]["b2_blocks"] = np.stack([m_con, m_non, g64, g128], 0)                            # 4 x n x 15
    C["F"]["b2_extra"] = np.c_[n64, n128]
    del A10, A30, CON30, mean, std, mx, hop2
    gc.collect()
    print(f"{key}: kNN-10 / 30 µm / contact / non-contact graphs built; mean contact degree {np.diff(C['CON'].indptr).mean():.2f}")
tick("graphs and neighbour features")

def table_nbr(C, drop=None):
    """Table + nbr (cell-type-graph's 78 inputs); drop = base column index to remove from every block (imputation)."""
    keep = [j for j in range(15) if j != drop]
    B = C["F"]["nbr_blocks"]
    return np.concatenate([C["F"]["base"][:, keep]] + [B[i][:, keep] for i in range(4)] + [C["F"]["nbr_extra"]], 1)

def b2_safe(C, drop=None):
    keep = [j for j in range(15) if j != drop]
    B = C["F"]["b2_blocks"]
    return np.concatenate([B[i][:, keep] for i in range(4)] + [C["F"]["b2_extra"]], 1)

def pix_cols(C, kind, drop_chan=None):
    """Pixel (or band) feature columns, without the target channel's columns."""
    names = C["names"][kind]
    keep = [i for i, nm in enumerate(names) if drop_chan is None or not nm.endswith(f"__{drop_chan}")]
    return C["F"][kind][:, keep]

In [ ]:
def r4_scorer(C, te, seed):
    tp, tcode = np.unique(C["patient"][te], return_inverse=True)
    W = np.random.default_rng(seed).multinomial(len(tp), np.full(len(tp), 1 / len(tp)), size=N_BOOT).astype(np.float64)
    img = C["img"][te]
    def f(a, b):
        ok = np.isfinite(a) & np.isfinite(b)
        a, b, im, pc = a[ok].astype(np.float64), b[ok].astype(np.float64), img[ok], tcode[ok]
        k = int(img.max()) + 1
        cnt = np.maximum(np.bincount(im, minlength=k), 1)
        a = a - (np.bincount(im, a, k) / cnt)[im]
        b = b - (np.bincount(im, b, k) / cnt)[im]
        sxy, sxx, syy = (np.bincount(pc, a * b, len(tp)), np.bincount(pc, a * a, len(tp)), np.bincount(pc, b * b, len(tp)))
        return float(sxy.sum() / np.sqrt(sxx.sum() * syy.sum())), (W @ sxy) / np.sqrt((W @ sxx) * (W @ syy))
    return f

def f1_scorer(C, imgs_cells, task, ncls, seed):
    node = imgs_cells[C["y"][task][imgs_cells] >= 0]
    pats, pc = np.unique(C["patient"][node], return_inverse=True)
    W = np.random.default_rng(seed).multinomial(len(pats), np.full(len(pats), 1 / len(pats)), size=N_BOOT).astype(np.float64)
    y = C["y"][task][node]
    present = np.bincount(y, minlength=ncls) > 0
    def metrics(M):
        M = M.reshape(-1, ncls, ncls)
        tp_ = np.einsum("bii->bi", M)
        sup, prd = M.sum(2), M.sum(1)
        with np.errstate(invalid="ignore", divide="ignore"):
            f1 = np.where(sup + prd > 0, 2 * tp_ / (sup + prd), 0.0)
        return f1[:, present].mean(1), f1
    def f(pred_node):
        cm = np.bincount((pc * ncls + y) * ncls + pred_node, minlength=len(pats) * ncls * ncls).reshape(len(pats), ncls * ncls).astype(np.float64)
        pt, f1pt = metrics(cm.sum(0, keepdims=True))
        bt, _ = metrics(W @ cm)
        return float(pt[0]), bt, f1pt[0]
    return node, f

def ci(v):
    v = np.asarray(v)[np.isfinite(v)]
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

def cells_of(C, imgs):
    st = C["starts"]
    return np.concatenate([np.arange(st[i], st[i + 1]) for i in imgs]) if len(imgs) else np.zeros(0, np.int64)

D_TRVA_PAT = np.unique(CD["patient"][CD["split0"] != "test"])

def make_splits(REP):
    jp, jcode = np.unique(CJ["patient"], return_inverse=True)
    u = np.random.default_rng(SEED + 11 + REP).random(len(jp))
    J_SPLIT = np.where(u < 0.7, "train", np.where(u < 0.8, "val", "test"))[jcode]
    uv = np.random.default_rng(SEED + 21 + REP).random(len(D_TRVA_PAT))
    dval = set(D_TRVA_PAT[uv < 0.125])
    D_SPLIT = np.where(CD["split0"] == "test", "test", np.where(pd.Series(CD["patient"]).isin(dval).values, "val", "train"))
    out = {}
    for k, s in (("Jackson", J_SPLIT), ("Danenberg", D_SPLIT)):
        C = CO[k]
        si = s[C["starts"][:-1]]
        out[k] = {t: np.flatnonzero(si == t) for t in ("train", "val", "test")}
    return out

In [ ]:
def class_weights(y, ncls):
    cnt = np.bincount(y, minlength=ncls).astype(np.float64)
    w = np.where(cnt > 0, 1 / np.sqrt(np.maximum(cnt, 1)), 0.0)
    return w / w[y].mean()

def fit_clf(X, y, ncls, seed, n_est=None):
    prm = dict(LGB)
    if n_est:
        prm["n_estimators"] = n_est
    return lgb.LGBMClassifier(random_state=seed, **prm).fit(X, y, sample_weight=class_weights(y, ncls)[y])

def proba_full(mdl, X, ncls):
    P_ = np.zeros((len(X), ncls), np.float32)
    for s in range(0, len(X), 200_000):
        P_[s:s + 200_000][:, mdl.classes_] = mdl.predict_proba(X[s:s + 200_000])
    return P_

def stage1(src, SPL, seed):
    """Out-of-fold (5 folds by image) coarse-type probabilities on the source training images; a full model elsewhere."""
    C = CO[src]
    XT = {k: table_nbr(CO[k]) for k in CO}
    tr_imgs = SPL[src]["train"]
    folds = np.random.default_rng(seed).permutation(len(tr_imgs)) % 5
    rs = np.random.default_rng(seed + 1)
    PR = {k: np.zeros((CO[k]["n"], len(COARSE)), np.float32) for k in CO}
    def fit_on(imgs):
        idx = cells_of(C, imgs)
        idx = idx[C["y"]["coarse"][idx] >= 0]
        idx = np.sort(rs.choice(idx, min(N_FIT_S1, len(idx)), replace=False))
        return fit_clf(XT[src][idx], C["y"]["coarse"][idx], len(COARSE), seed, n_est=60 if SMOKE else 150)
    for f in range(5):
        mdl = fit_on(tr_imgs[folds != f])
        idx = cells_of(C, tr_imgs[folds == f])
        PR[src][idx] = proba_full(mdl, XT[src][idx], len(COARSE))
    mdl = fit_on(tr_imgs)
    for k in CO:
        other = np.setdiff1d(np.arange(CO[k]["n_img"]), tr_imgs) if k == src else np.arange(CO[k]["n_img"])
        idx = cells_of(CO[k], other)
        PR[k][idx] = proba_full(mdl, XT[k][idx], len(COARSE))
    del XT
    return PR

def b2_pred(C, PRk):
    """Predicted-type mixtures over contact / non-contact neighbours and 64 / 128 µm patches, + contact counts by predicted type."""
    m_con, _ = row_mean(C["A_con"], PRk)
    m_non, _ = row_mean(C["A_non"], PRk)
    g64, _ = grid_mean(C, PRk, 64.0)
    g128, _ = grid_mean(C, PRk, 128.0)
    hard = np.eye(len(COARSE), dtype=np.float32)[PRk.argmax(1)]
    cnt_con = C["A_con"] @ hard
    return np.c_[m_con, m_non, g64, g128, np.log1p(cnt_con)].astype(np.float32)

## Count features

In [ ]:
for C in CO.values():
    G30 = ((C["A_con"] + C["A_non"]) > 0).astype(np.float32).tocsr()
    C["G"] = {"contact": C["A_con"], "30um": G30}

def count_feats(C, lab, known, ncls):
    """lab: label per cell (-1 = no label); known: cells whose label may be used. With / without / known counts per graph."""
    oh = np.zeros((C["n"], ncls), np.float32)
    ok = known & (lab >= 0)
    oh[np.flatnonzero(ok), lab[ok]] = 1.0
    kn = known.astype(np.float32)[:, None]
    out = []
    for G in C["G"].values():
        w = G @ oh
        n_known = G @ kn
        out += [w, n_known - w, n_known]
    return np.concatenate(out, 1).astype(np.float32)

def mix_feats(C, PRk):
    out = []
    for G in C["G"].values():
        m, _ = row_mean(G, PRk)
        out.append(m)
    return np.concatenate(out, 1).astype(np.float32)

def pred_feats(C, PRk, with_mix):
    f = count_feats(C, PRk.argmax(1).astype(np.int64), np.ones(C["n"], bool), len(COARSE))
    return np.c_[f, mix_feats(C, PRk)] if with_mix else f

def reveal(C, frac, seed):
    return np.random.default_rng(seed).random(C["n"]) < frac

## Replicates

In [ ]:
UNITS = [("coarse", "Jackson", "Jackson"), ("coarse", "Danenberg", "Danenberg"), ("coarse", "Jackson", "Danenberg"),
         ("coarse", "Danenberg", "Jackson"), ("fine", "Danenberg", "Danenberg"), ("fine", "Jackson", "Jackson")]
FRACS = {"coarse": [0.05, 0.2, 0.5], "fine": [0.2]}
ROWS, DIFF, LOG = [], [], []

def stage2(src, SPL, PR1, seed):
    """Out-of-fold A2 (coarse) probabilities on the source training images, a full A2 model elsewhere: the round-2 inputs."""
    C = CO[src]
    XA = {k: np.c_[table_nbr(CO[k]), pred_feats(CO[k], PR1[k], True)] for k in CO}
    tr_imgs = SPL[src]["train"]
    folds = np.random.default_rng(seed).permutation(len(tr_imgs)) % 5
    rs = np.random.default_rng(seed + 1)
    PR = {k: np.zeros((CO[k]["n"], len(COARSE)), np.float32) for k in CO}
    def fit_on(imgs):
        idx = cells_of(C, imgs)
        idx = idx[C["y"]["coarse"][idx] >= 0]
        idx = np.sort(rs.choice(idx, min(N_FIT_S1, len(idx)), replace=False))
        return fit_clf(XA[src][idx], C["y"]["coarse"][idx], len(COARSE), seed, n_est=60 if SMOKE else 150)
    for f in range(5):
        mdl = fit_on(tr_imgs[folds != f])
        idx = cells_of(C, tr_imgs[folds == f])
        PR[src][idx] = proba_full(mdl, XA[src][idx], len(COARSE))
    mdl = fit_on(tr_imgs)
    for k in CO:
        other = np.setdiff1d(np.arange(CO[k]["n_img"]), tr_imgs) if k == src else np.arange(CO[k]["n_img"])
        idx = cells_of(CO[k], other)
        PR[k][idx] = proba_full(mdl, XA[k][idx], len(COARSE))
    return PR

def record(REP, task, src, tst, arm, setting, base_bt, pt, bt, base_pt):
    lo, hi = ci(bt)
    ROWS.append(dict(rep=REP, task=task, train=src, test=tst, arm=arm, setting=setting, macro_f1=pt, lo=lo, hi=hi))
    if base_bt is not None:
        dlo, dhi = ci(bt - base_bt)
        DIFF.append(dict(rep=REP, task=task, train=src, test=tst, arm=arm, setting=setting, diff=pt - base_pt, lo=dlo, hi=dhi))

for REP in REPS:
    t_rep = time.time()
    tick(f"replicate {REP}")
    SPL = make_splits(REP)
    TE = {k: cells_of(CO[k], SPL[k]["test"]) for k in CO}
    PR1 = {s: stage1(s, SPL, SEED + 61 + REP) for s in ("Jackson", "Danenberg")}       # same seeds as multiscale-audit
    PR2 = {s: stage2(s, SPL, PR1[s], SEED + 71 + REP) for s in ("Jackson", "Danenberg")}
    tick(f"rep {REP}: stage-1 and stage-2 predicted types")
    for task, src, tst in UNITS:
        if (time.time() - T0) / 60 > BUDGET_MIN:
            print(f"!!! skipping rep {REP} {task} {src}->{tst}")
            continue
        C, Ct = CO[src], CO[tst]
        ncls = len(COARSE) if task == "coarse" else len(C["fine"])
        y_tr, y_te = C["y"][task], Ct["y"][task]
        idx = cells_of(C, SPL[src]["train"])
        idx = idx[y_tr[idx] >= 0]
        fit = np.sort(np.random.default_rng(SEED + REP).choice(idx, min(N_FIT_TYP, len(idx)), replace=False))
        node, f = f1_scorer(Ct, TE[tst], task, ncls, SEED + 41 + REP + 1000 * (tst == "Danenberg"))
        # arm 0 (same model as multiscale-audit arm 0)
        mdl0 = fit_clf(table_nbr(C)[fit], y_tr[fit], ncls, SEED + REP)
        P0 = proba_full(mdl0, table_nbr(Ct), ncls)
        pred0 = P0.argmax(1)
        pt0, bt0, _ = f(pred0[node])
        record(REP, task, src, tst, "0 base", "main", None, pt0, bt0, None)
        # arm A
        armsA = {"A1 pred counts": (lambda k, s: pred_feats(CO[k], PR1[s][k], False)),
                 "A2 pred counts + mix": (lambda k, s: pred_feats(CO[k], PR1[s][k], True))}
        if task == "coarse":
            armsA["A3 round 2"] = lambda k, s: pred_feats(CO[k], PR2[s][k], True)
        for arm, fx in armsA.items():
            t0 = time.time()
            mdl = fit_clf(np.c_[table_nbr(C), fx(src, src)][fit], y_tr[fit], ncls, SEED + REP)
            Xt = np.c_[table_nbr(Ct)[node], fx(tst, src)[node]]
            pred = mdl.classes_[mdl.predict_proba(Xt).argmax(1)]
            pt, bt, _ = f(pred)
            record(REP, task, src, tst, arm, "main", bt0, pt, bt, pt0)
            LOG.append(dict(rep=REP, task=task, train=src, test=tst, arm=arm, seconds=time.time() - t0))
        # arm B: revealed true labels (hidden cells only, paired with arm 0 on the same hidden cells)
        for frac in FRACS[task]:
            t0 = time.time()
            rv_tr = reveal(C, frac, SEED + 81 + REP)
            rv_te = rv_tr if tst == src else reveal(Ct, frac, SEED + 91 + REP)
            if tst == src:
                rv_te = rv_tr
            Ftr = count_feats(C, y_tr, rv_tr, ncls)
            Fte = Ftr if tst == src else count_feats(Ct, y_te, rv_te, ncls)
            fitB = fit[~rv_tr[fit]]
            mdl = fit_clf(np.c_[table_nbr(C)[fitB], Ftr[fitB]], y_tr[fitB], ncls, SEED + REP)
            hidden = TE[tst][~rv_te[TE[tst]]]
            nodeB, fB = f1_scorer(Ct, hidden, task, ncls, SEED + 51 + REP)
            ptb0, btb0, _ = fB(pred0[nodeB])
            predB = mdl.classes_[mdl.predict_proba(np.c_[table_nbr(Ct)[nodeB], Fte[nodeB]]).argmax(1)]
            pt, bt, _ = fB(predB)
            setting = f"revealed {int(frac * 100)}%"
            record(REP, task, src, tst, "0 base (hidden cells)", setting, None, ptb0, btb0, None)
            record(REP, task, src, tst, "B revealed counts", setting, btb0, pt, bt, ptb0)
            LOG.append(dict(rep=REP, task=task, train=src, test=tst, arm=f"B {frac}", seconds=time.time() - t0))
        tick(f"rep {REP} {task} {src}->{tst}: base {pt0:.3f}; " +
             ", ".join(f"{r['arm']} [{r['setting']}] {r['macro_f1']:.3f}" for r in ROWS[-(len(armsA) + 2 * len(FRACS[task])):]))
        pd.DataFrame(ROWS).to_csv(f"{OUT}/scores.csv", index=False)
        pd.DataFrame(DIFF).to_csv(f"{OUT}/diffs.csv", index=False)
        pd.DataFrame(LOG).to_csv(f"{OUT}/training_log.csv", index=False)
    del PR1, PR2
    gc.collect()
    tick(f"replicate {REP} done ({(time.time() - t_rep) / 60:.0f} min)")

## Summary, decisions, and the check against `multiscale-audit` arm 0

In [ ]:
R = pd.DataFrame(ROWS)
D = pd.DataFrame(DIFF)
n_reps = R.rep.nunique()
agg = R.groupby(["task", "train", "test", "setting", "arm"]).macro_f1.agg(["mean", "std", "count"]).reset_index()
agg.to_csv(f"{OUT}/summary_scores.csv", index=False)
print(agg.round(4).to_string(index=False))
dsum = D.groupby(["task", "train", "test", "setting", "arm"]).agg(
    mean_diff=("diff", "mean"), n_ci_pos=("lo", lambda v: int((v > 0).sum())), n_ci_neg=("hi", lambda v: int((v < 0).sum())),
    n=("diff", "size")).reset_index()
dsum["claim"] = (dsum.n == 3) & (dsum.n_ci_pos == 3) & (dsum.mean_diff >= 0.01)
dsum.to_csv(f"{OUT}/diff_summary.csv", index=False)
print(dsum.round(4).to_string(index=False))
ref = [p for p in g("typing_scores.csv") if "multiscale-audit" in p]
if ref and not SMOKE:
    T = pd.read_csv(ref[0])
    T = T[T.arm == "0 base"].rename(columns={"macro_f1": "audit"})[["rep", "task", "train", "test", "audit"]]
    M = R[(R.arm == "0 base") & (R.setting == "main")][["rep", "task", "train", "test", "macro_f1"]]
    chk = T.merge(M, on=["rep", "task", "train", "test"])
    chk["diff"] = chk.macro_f1 - chk.audit
    print(chk.round(5).to_string(index=False))
    chk.to_csv(f"{OUT}/check_vs_multiscale_audit.csv", index=False)
else:
    print("multiscale-audit output not mounted (or smoke run): arm-0 reproduction check skipped")
json.dump({"n_reps": int(n_reps), "claims": dsum[dsum.claim][["task", "train", "test", "setting", "arm", "mean_diff"]].to_dict("records"),
           "smoke": SMOKE, "minutes": (time.time() - T0) / 60}, open(f"{OUT}/decision.json", "w"), indent=1)